NOTEBOOK DIFFERS FROM PY FILES IN THE IMPORTS (CELL TITLE "SETUP") AND THE FOV SELECTION CODE (CELL TITLE "CROPPING FUNCTIONS")

In [ ]:
#@title setup

!pip install dash #for interactive plots
!pip install jupyter-dash

import shutil
from skimage import io
import mat73

from deepcad.train_collection import training_class
from deepcad.test_collection import testing_class
from deepcad.movie_display import display, display_img
from deepcad.utils import get_first_filename

import caiman as cm
import caiman.source_extraction.cnmf as cnmf

import cv2
import datetime
import fnmatch
import logging
from ast import literal_eval
import sys
import re
import numpy as np
from scipy import signal
import scipy.io as sio
import random
from tqdm import tqdm
import scipy
import glob
import os
from tifffile.tifffile import imwrite, imread

from ScanImageTiffReader import ScanImageTiffReader
from numpy.core.records import fromarrays

import matplotlib.pyplot as plt
from matplotlib.widgets  import RectangleSelector

from skimage.util.dtype import img_as_float
import plotly.express as px
import dash_core_components as dcc
import dash_html_components as html
from dash.dependencies import Input, Output
from jupyter_dash import JupyterDash
import time

import bokeh.plotting as bpl
bpl.output_notebook()

# try:
#     cv2.setNumThreads(0)
# except:
#     pass

ipyparalell?

try:
    if __IPYTHON__:
        # this is used for debugging purposes only. allows to reload classes when changed
        get_ipython().magic('load_ext autoreload')
        get_ipython().magic('autoreload 2')
except NameError:
    pass

logging.basicConfig(format=
                          "%(relativeCreated)12d [%(filename)s:%(funcName)20s():%(lineno)s] [%(process)d] %(message)s",
                    # filename="/tmp/caiman.log",
                    #level=logging.INFO,
                    )





In [ ]:
def read_save_metadata(pth_datafile, pth_md):

    # use ScanImageTiffReader to read metadata (strange parsing because scanimage tif headers are not saved as json)
    meta = ScanImageTiffReader(pth_datafile).metadata()   
    mdt = {}
    mdt['numvol'] = int(re.findall( 'actualNumVolumes = (.*)', meta)[0])
    mdt['numslice_withflyback'] = int(re.findall( 'numFramesPerVolumeWithFlyback = (.*)', meta)[0])
    mdt['numslice'] = int(re.findall( 'actualNumSlices = (.*)', meta)[0])
    mdt['xpix'] = int(re.findall( 'pixelsPerLine = (.*)', meta)[0])
    mdt['ypix'] = int(re.findall( 'linesPerFrame = (.*)', meta)[0])
    mdt['flyback'] = mdt['numslice_withflyback'] - mdt['numslice']
    mdt['dims'] = [mdt['numvol'], mdt['numslice_withflyback'] - mdt['flyback'], mdt['ypix'], mdt['xpix']]
    fovtmp = literal_eval(re.findall( 'imagingFovUm = (.*)', meta)[0].replace(" ",",").replace(";",","))
    mdt['xfov'] = abs(fovtmp[0]) + abs(fovtmp[2])
    mdt['yfov'] = abs(fovtmp[1]) + abs(fovtmp[3])
    mdt['zwid'] = int(re.findall( 'actualStackZStepSize = (.*)', meta)[0])
    mdt['zstartpos'] = literal_eval(re.findall( 'zsRelative = (.*)', meta)[0].replace(";",","))
    mdt['zfov'] = mdt['zstartpos'][-1] + mdt['zwid'] - mdt['zstartpos'][0]
    mdt['framerate'] = float(re.findall( 'scanFrameRate = (.*)', meta)[0])
    mdt['volrate'] = float(re.findall( 'scanVolumeRate = (.*)', meta)[0])

    md = fromarrays( [ mdt['numvol'], mdt['numslice_withflyback'], mdt['numslice'], mdt['xpix'], \
        mdt['ypix'], mdt['flyback'], mdt['xfov'], mdt['yfov'], \
        mdt['zwid'], mdt['zfov'], mdt['framerate'], mdt['volrate'] ], \
        names = ['numvol', 'numslice_withflyback', 'numslice', 'xpix', 'ypix', 'flyback', \
            'xfov', 'yfov', 'zwid', 'zfov', 'framerate', 'volrate' ] )

    sio.savemat(pth_md[0], {'md': md}) #save for matlab part of pipeline 
    
    return mdt


In [ ]:
#@title map2params

class map2params():
    def __init__(self):

        self.params = {}
        self.params['m2p_merge_thresh'] = [.9]
        self.params['m2p_gsig'] = [2] #[2, 3, 4]
        self.params['m2p_nb'] = [1, 2]
        self.params['SC_sigma'] = [1, 2, 4]
        self.params['lambda_gnmf'] = [1, 2, 4]
        self.params['perc_baseline_snmf'] = [20] #10, 20, 40]
        self.params['max_iter_snmf'] = [1000]

        self.map = []
        self.max_index = 1
        for key,val in self.params.items():
            self.outer(val)
            self.max_index *= len(val)
        self.max_index-= 1
        self.order = list(self.params.keys())

    def outer(self,l2):
        if self.map == []:
            self.map = [[f2] for f2 in l2]
        else:
            prod = []
            for f1 in self.map:
                for f2 in l2:
                    prod.append(f1+[f2])
            self.map = prod
        return None

    def map_index(self,index):
        assert 0 <= index <= self.max_index
        return self.map[index]


In [ ]:
#@title bgremover

class bgremover:

    def __init__(self, img, pth_save_prefix, half_wid=12, dimorder='tyx'):
        self.half_wid = half_wid
        self.pth_save_prefix = pth_save_prefix
        if dimorder=='tyx':
            self.img = img
        elif dimorder=='txy':
            self.img = np.transpose(img, (0, 2, 1))

    def draw_patches(self):
        half_wid = self.half_wid
        wid = 2*half_wid
        kernel = np.ones(wid)/wid
        self.meanframe = np.mean(self.img, axis=0)
        bg_ind = []
        for line in self.meanframe:
            tmp = np.convolve(line, kernel, 'valid')
            bg_center = np.argmin(tmp) + half_wid
            bg_ind.append([bg_center-half_wid, bg_center+half_wid])
        self.bg_ind = bg_ind

    def remove_bg(self, offset=0):
        bg_ind = self.bg_ind
        out = self.img.copy()
        for ind in range(out.shape[1]):
            patch = self.img[:, ind, :]
            bg_patch = self.img[:, ind, bg_ind[ind][0]:bg_ind[ind][1]]
            bg = bg_patch.mean(axis=-1)
            patch = patch-bg[None].T
            out[:, ind, :] = patch
        out = out + offset # compensate so that most of the pixels are above 0 (deprecated)
        self.out = out

    def make_plots(self):

        half_wid = self.half_wid
        
        patch_im = np.mean(self.img, axis=0)
        mv = np.max(patch_im)
        for i in range(patch_im.shape[0]):
            patch_im[i, self.bg_ind[i][0]:self.bg_ind[i][1]] = mv
        
        y_len, x_len = self.out.shape[1:3]
        test_y = random.randint(0, y_len-1)
        test_x = random.randint(0, x_len-2*half_wid-1) + half_wid

        test_patch = self.img[:, test_y, test_x-half_wid:test_x+half_wid]
        test = test_patch.mean(-1)
        test = test/test.mean()

        fs = self.img.shape[-1]
        f, Pxx_den = signal.periodogram(test, fs)
        fig, ((ax1, ax2), (ax3, ax4), (ax5, ax6), (ax7, ax8)) = plt.subplots(4, 2)
        ax1.semilogy(f[1:], Pxx_den[1:]) #skip 0 frequency

        test_patch = self.out[:, test_y, test_x-half_wid:test_x+half_wid]
        test = test_patch.mean(-1)
        test = test/test.mean()

        f, Pxx_den = signal.periodogram(test, fs)
        ax2.semilogy(f[1:], Pxx_den[1:]) #skip 0 frequency

        ax3.imshow(self.meanframe)
        ax3.axis('off')
        
        self.meanframe_nobg = np.mean(self.out, axis=0)

        ax4.imshow(self.meanframe_nobg)
        ax4.axis('off')

        ax5.hist(self.img.ravel(), bins=50)
        ax5.set_yscale('log')
        ax6.hist(self.out.ravel(), bins=50)
        ax6.set_yscale('log')
            
        ax7.imshow(patch_im)

        plt.savefig(self.pth_save_prefix + '_bgplots.png')
        plt.close()



In [ ]:
#@title configs

def configs(index_extraction_param_set = None, fnames = None, min_mov = 0,  md = None, do_planar_extraction = None, dims_spatial = (1,1,1)):


    #motion correction configs
    strides_mc = (24, 24, 6)
    overlaps_mc = (12, 12, 3)
    max_shifts_mc = (4, 4, 2)
    max_deviation_rigid = 3
    pw_rigid = False
    is3D_mc = True
    nonneg_movie = True
    min_mov = min_mov
    shifts_opencv = True
    indices_mc = (slice(None), slice(None), slice(None)) #if is3d is true for motion correction, will overwrite with nones and will lose indices_ex

    do_slices = False #my crop_fov is meant to replace this, so should always be false
    if do_slices:
        sly = slice(6, 51, 1)
        slx = slice(40, 181, 1)
        slz = slice(1, 9, 1)
        indices_ex = [slx, sly, slz]
    else:
        indices_ex = [slice(None), slice(None), slice(None)]


    only_init = False #only use the initialization run for extraction

    p = 0                   # order of the autoregressive system - 0 for nonspiking, 1 for instanteous rise but not decay, 2 for non-ionstantaneous rise and decay
    merge_thresh = 0.9
    gSig = [2, 2, 1] #forces to be odd so gsiz min is 3 (ie gsig 0.5 is same as 1)  # gSig = [3,3]            # radius (half-size) of average neurons (in pixels)
    nb = 1 #num background components

    fr = md['volrate'] #0.6193  #9.8465 frame period so 1000 / (9.8465 *(113+51)) # approximate frame rate of data - CONFIRMED FPS
    decay_time = .4         # length of transient - CONFIRMED APPROPRIATE FOR OUR INDICATOR GCaMP6f
    dxy = [md['xpix']/md['xfov'], md['ypix']/md['yfov'], md['numslice']/md['zfov']] #pixels per micron

    tsub = 1                # temporal downsampling
    ssub = 1               # spatial downsampling
    p_ssub = 1 #patch downsampling in space
    p_tsub = 1 #patch downsampling in time

    thr_method = 'nrg' #or 'max'
    maxthr = 0.1 #for  thr_method = 'max' keep pixels above this threshold
    nrgthr = 0.9999 #for  thr_method = 'nrg' keep pixels whose sorted cumsum contributes this much of total energy
    extract_cc = True #true will throw away isolated pixels of some kind

    method_init = 'sparse_nmf' #'greedy_roi' #'graph_nmf' #sparse_NMF apparently has problems?? 'greedy_roi' python Caiman defaults to greedy_roi, looks for globular sources

    max_iter_snmf = 1000
    perc_baseline_snmf = 20
    alpha_snmf = 100 #default 1000 #for method_init sparseNMF
    #sigma_smooth_snmf = gSig #(2, 2, 0.5) #default 0.5 0.5 0.5

    lambda_gnmf = 1 #for method_init graphNMF
    SC_kernel = 'heat' #NOT TUNABLE NOW       # kernel for graph affinity matrix
    SC_sigma = 1              # std for SC kernel
    SC_thr = 0                 # threshold for affinity matrix
    SC_normalize = True        # standardize entries prior to computing affinity matrix
    SC_use_NN = False          # sparsify affinity matrix by using only nearest neighbors
    SC_nnn = 20                # number of nearest neighbors to use if SC_use_NN = True


    low_rank_background = True #true makes bankground nb, false makes it update with hals
    update_background_components = True   #use this??

    fudge_factor = 0.96        # (default is 0.96; old value = 1) -- bias correction factor for discrete time constants
    ITER = 5                # (default is 2; old value=5) -- block coordinate descent iterations
    bas_nonneg = False #True

    rolling_sum = True

    #Each parameter has a low threshold (rval_lowest (default -1), SNR_lowest (default 0.5), cnn_lowest (default 0.1))
    # and high threshold (rval_thr (default 0.8), min_SNR (default 2.5), min_cnn_thr (default 0.9)).
    # A component has to exceed ALL low thresholds as well as ONE high threshold to be accepted.
    # can turn off CNN part withy use_CNN = false
    # these values will result in no roi filtering
    SNR_lowest = 0#0.5#0  #0.5 default       # minimum SNR for accepted components
    min_SNR = 0#0  #2.5 default    # accept components with that peak-SNR or higher
    rval_lowest = -1  # -1 default 0.6  # space correlation threshold
    rval_thr = 0#0  # 0.8 default  # space correlation threshold
    use_cnn = False      # True default # use the CNN classifier affects if 2 below params are used
    cnn_lowest = 0  #0.1 default  # neurons with cnn probability lower than this value are rejected
    min_cnn_thr = 0 #0.9 default # if cnn classifier predicts below this value, reject

    stride_to_rf_ratio = 0.65

    if index_extraction_param_set is not None: #create param set whose index matches value in index_extraction_param_set
        map_index_2_params = map2params()
        index_extraction_param_set = int(index_extraction_param_set)
        print('reading parameters from the index_extraction_param_set file')
        merge_thresh, m2p_gsig, nb, SC_sigma, lambda_gnmf, perc_baseline_snmf, max_iter_snmf = map_index_2_params.map_index(index_extraction_param_set)
        gSig = [m2p_gsig, m2p_gsig, 1]  #gSiz (made from gsig) will be 2 for 0.5 or 1, so don't bother with 0.5

    if dims_spatial[0]<50 and dims_spatial[1]<50 and dims_spatial[2]<50: #dont bother with patches if FOV is small enough (but this should be adjusted for dirtier drivers)
        do_patches = False
    else:
        do_patches = True

    #determine k in automated way based on gSig, roi_decimation_fac, and stride_to_rf_ratio, while also satisfying caiman patch size recommendations
    roi_decimation_fac = 0.4
    if do_patches: # PROCESS IN PATCHES AND THEN COMBINE, patches are useful if activity stats vary over fov (e.g. extracting same neurons from regions with varying SNR, patch runs will adapt to local stats)
        rf = int(np.ceil((np.max(gSig)*2+1) / stride_to_rf_ratio)) + 1
        stride_cnmf = int(np.round(rf * stride_to_rf_ratio))         # overlap between patches
        p_patch = p
        nb_patch = nb
        if do_planar_extraction==True:
            k = int(np.round( (rf*2*rf*2) / ((gSig[0]*2+1)*(gSig[1]*2+1))*roi_decimation_fac))  # number of components in each patch, rf and gsig are both half sizes
            #k = int(np.round(rf*rf / (gSig[0]*gSig[1])))
        else:
            if rf*2<dims_spatial[-1]:
                rfz = rf*2
            else:
                rfz = dims_spatial[-1]
            k = int(np.round( (rf*2*rf*2*rfz) / ((gSig[0]*2+1)*(gSig[1]*2+1)*(gSig[2]*2+1))*roi_decimation_fac))  # number of components in each patch, rf and gsig are both half sizes
        indices_ex = [slice(None), slice(None), slice(None)]
    else:                   # PROCESS THE WHOLE FOV AT ONCE
        rf = None # will run CNMF on the whole FOV
        stride_cnmf = None       # will run CNMF on the whole FOV
        p_patch = p
        nb_patch = nb
        k = int(np.round( np.prod(dims_spatial) / ((gSig[0]*2+1)*(gSig[1]*2+1)*(gSig[2]*2+1))*roi_decimation_fac))  # number of components in each patch, rf and gsig are both half sizes

    dimstr = "3dex_"
    if do_planar_extraction==True:
        dimstr = '2dex_'
        indices_ex = indices_ex[:-1] #change from 3d to 2d
        dxy = dxy[:-1] #change from 3d to 2d
        gSig = gSig[:-1] #change from 3d to 2d

    gSiz = [int(np.round(2*gstmp + 1)) for gstmp in gSig] #put here at end to register any gSig change
    se = np.ones((3,)*len(gSig), dtype=np.uint8)  #put here at end to register any gSig change #se = np.ones((3,3,1), dtype=np.uint8)
    #medw = (3,)*len(gSig)

    sigma_smooth_snmf = [0.5] #append this filter sigma for time to beginning (when it is applied time is in 1st dim?)
    sigma_smooth_snmf.extend(gSig)
    if do_planar_extraction==False:
        sigma_smooth_snmf[-1] = 0.5 #sigma_smooth_snmf can actually use values<1, if 3d extraction, make small for coarse z samples

    fnadd = '_' + str(gSig[0]) + '_' + str(nb) + '_' + str(merge_thresh) + '_' + str(k) + \
        '_' + str(rf) + '_' + str(SC_sigma) + '_' + str(lambda_gnmf) + '_' + str(perc_baseline_snmf) \
            + '_' + str(max_iter_snmf) + '_' + method_init.split('_')[0] + '_' + dimstr

    if do_planar_extraction is None: #it's none during motion correction, when we don't care about these params, rather than true/false
        print("motion correction params configured")
    else:
        print("index_extraction_param_set is " + str(index_extraction_param_set) + " with filename string " + fnadd)

    opts_dict = {'strides': strides_mc,    # start a new patch for pw-rigid motion correction every x pixels
                'overlaps': overlaps_mc,   # overlap between pathes (size of patch strides+overlaps)
                'max_shifts': max_shifts_mc,   # maximum allowed rigid shifts (in pixels)
                'max_deviation_rigid': max_deviation_rigid,  # maximum shifts deviation allowed for patch with respect to rigid shifts
                'pw_rigid': pw_rigid,         # flag for performing non-rigid motion correction
                'is3D': is3D_mc,
                'nonneg_movie':nonneg_movie,
                'min_mov': min_mov,
                'shifts_opencv':shifts_opencv,
                'fr': fr,
                'p': p,
                'nb': nb,
                'merge_thr': merge_thresh,
                'rf': rf,
                'indices': indices_mc,  #for some reason indices_mc is causing error, maybe needs list for mc and tuple for extraction?
                'K': k,
                'gSig': gSig,
                'gSiz': gSiz,
                'stride': stride_cnmf,
                'method_init': method_init,
                'sigma_smooth_snmf': sigma_smooth_snmf,
                'perc_baseline_snmf': perc_baseline_snmf,
                'max_iter_snmf': max_iter_snmf,
                'sigma_smooth_snmf': sigma_smooth_snmf,
                'SC_kernel': SC_kernel, #NOT TUNABLE NOW       # kernel for graph affinity matrix
                'SC_sigma': SC_sigma,            # std for SC kernel
                'SC_thr': SC_thr,                 # threshold for affinity matrix
                'SC_normalize': SC_normalize,        # standardize entries prior to computing affinity matrix
                'SC_use_NN': SC_use_NN,          # sparsify affinity matrix by using only nearest neighbors
                'SC_nnn': SC_nnn,                # number of nearest neighbors to use if SC_use_NN = True
                'lambda_gnmf': lambda_gnmf, #for method_init graphNMF
                'alpha_snmf': alpha_snmf, #for method_init sparseNMF
                #'dims': dims,
                'dxy': dxy,
                'decay_time': decay_time,
                'bas_nonneg': bas_nonneg,
                'p_patch': p_patch,
                'nb_patch': nb_patch,
                'se': se,
                'rolling_sum': rolling_sum,
                'only_init': only_init,
                'ssub': ssub,
                'tsub': tsub,
                'p_ssub': p_ssub,
                'p_tsub': p_tsub,
                'SNR_lowest': SNR_lowest,
                'min_SNR': min_SNR,
                'rval_thr': rval_thr,
                'rval_lowest': rval_lowest,
                'use_cnn': use_cnn,
                'min_cnn_thr': min_cnn_thr,
                'cnn_lowest': cnn_lowest,
                'ITER': ITER,
                'fudge_factor': fudge_factor,
                'cnn_lowest': cnn_lowest,
                'update_background_components': update_background_components,
                'low_rank_background' : low_rank_background,
                'thr_method': thr_method,
                'maxthr': maxthr,
                'nrgthr': nrgthr,
                'fnames': fnames,
                #'medw': medw,
                'extract_cc': extract_cc}


    # # for reference here are the initialization defs for 3 methods, sparse_nmf apparently "has problems" according to gitter

    # # greedyROI(Y, nr=30, gSig=[5, 5], gSiz=[11, 11], nIter=5, kernel=None, nb=1,
    #           rolling_sum=False, rolling_length=100, seed_method='auto')

    # for graphnmf all these are tunable in cnmf params except remove_baseline, truncate, tol, and SC_kernel whose defaults are below
    # note SC_kernel appears to be tunable because it's in params but it is not, heat is default
    # # graphNMF(Y_ds, nr, max_iter_snmf=500, lambda_gnmf=1,
    #          sigma_smooth=(.5, .5, .5), remove_baseline=True,
    #          perc_baseline=20, nb=1, truncate=2, tol=1e-3, SC_kernel='heat',
    #          SC_normalize=True, SC_thr=0, SC_sigma=1, SC_use_NN=False,
    #          SC_nnn=20

    # for sparsenmf all these are tunable in cnmf params except remove_baseline and truncate, whose defaults are below
    # # sparseNMF(Y_ds, nr, max_iter_snmf=500, alpha=10e2, sigma_smooth=(.5, .5, .5),
    #           remove_baseline=True, perc_baseline=20, nb=1, truncate=2)


    return opts_dict, indices_ex, fnadd



In [ ]:
#@title cropping functions

def select_fov_z(images):

  %matplotlib inline

  numim = images.shape[-1]

  Nc = 3
  Nr = int(np.ceil(numim/Nc))
  fig, ax = plt.subplots(Nr, Nc)
  for i in range(numim):
      indies = np.unravel_index(i, (Nr,Nc))
      ax[indies[0],indies[1]].imshow(images[:,:,i].T)
      ax[indies[0],indies[1]].axis('off')
  interval = 5 # units of seconds
  plt.pause(interval)
  plt.show()
  print("what z slices do you want to keep? consider keeping first as padding if cells abut z edges")
  zlimits = literal_eval(input ("choose z limits, format (firstframe,lastframe) one-indexed: "))
  return zlimits



def select_fov_xy(img):

  img = img.T
  fig = px.imshow(img)
  fig_hist = px.histogram(img.ravel())

  fig.update_layout(
      autosize=False,
      # width=500,
      height=400,
      dragmode='drawrect',
      newshape=dict(line_color='cyan', line_width=0.5))

  # this doesn't work: plots a second figure that is editable but lacks propertiers of the other
  # fig.show(config={
  #           'editable': True,
  #           'edits': {'shapePosition': True}})

  app = JupyterDash(__name__)
  app.layout = html.Div(
      [
          html.H3("Drag a rectangle to show the histogram of the ROI"),
          html.Div([dcc.Graph(id="middle-sai", figure=fig)], style={"width": "60%", "display": "inline-block", "padding": "0 0"}),
          html.Pre(id="bbox-data", style={"width": "10%", "display": "inline-block", "padding": "0 0"}),
          #html.Div(id='all-bboxes',style={"color": "white"}),
          html.Div([dcc.Graph(id="histogram", figure=fig_hist)], style={"width": "30%", "display": "inline-block", "padding": "0 0"}),
      ]
  )

  @app.callback(
      Output("histogram", "figure"),
      Output("bbox-data", "children"),
      Input("middle-sai", "relayoutData"),
      prevent_initial_call=True,
  )
  def on_new_annotation(relayout_data):

      if "shapes" in relayout_data:

          last_shape = relayout_data["shapes"][-1]
          x1, y0 = int(last_shape["x0"]), int(last_shape["y0"]) #slicing requires ints
          x0, y1 = int(last_shape["x1"]), int(last_shape["y1"]) #slicing requires ints
          roi_img = img[y0:y1, x0:x1]
          coords = (y0, y1, x0, x1)
          croplimxy = np.asarray(coords).astype(int)
          fn_crop_lim_xy = pth_tif_reg[0][:-4] + 'croplimxytmp_.npy'
          with open(fn_crop_lim_xy, 'wb') as fncrop:
              np.save(fncrop, croplimxy)
          return px.histogram(roi_img.ravel()),  html.Div(str(coords), style={"color": "white"})

      else:
          return no_update

  app.run()




In [ ]:
#@title choose files

index_extraction_param_set = 0 #specifies the extraction param set (set is created in configs.py, which uses map2params.py to help create the param sets)
recdates = ['20230627'] #list of strings, as it appears in the directory and raw file filename (with hyphen not underscore for now), '*' for any
fly = '2' #string, fly index_extraction_param_set, '*' for any
trial = '2' #string, trial index_extraction_param_set, '*' for any
region_extraction = ['pb', 'gar', 'gal', 'no'] #list of strings specifying names for xy rectangular or xyz cuboid fov subregions that are passed separately to source extraction; interactive plots prompt user to define z range and draw xy rectangle; use [''] to extract from entire FOV
do_background_subtraction = True
do_motion_correction = False #caiman normCorre
do_denoise = False #deepcad (from the more recent deepcadrt, although this is not real time), input must be motion_corrected
use_denoised = False #use the deepcad denoised data, or just the caiman registered data
do_extraction = True #caiman source extraction
do_planar_extraction = False #caiman source extraction for each plane independently (WARNING, 3D EXTRACTION REQUIRES AT LEAST 3 ELEMENTS IN EACH DIMENSION X Y and Z, OR you must REWRITE binary_closing IN CAIMAN'S THRESHOLD_COMPONENTS)
do_cropping_session = False #skip everything but FOV selection for all entries in region_extraction, must have already run motion correction if use_denoised=False, or motion correction and denoising if use_denoised=True, convenient to do for many recordings at once so extraction can be run on a batch of recordings in pth_allrecs without interruption
recording_index = 0 #if 0, loop over all recordings in pth_allrec, if not 0, operate on recording whose index (in sorted list of all recordings in pth_allrec) matches value in recording_index

bg_patch_halfwidth = 8
do_plots = 0 #plots were for old version of this pipeline, and I haven't verified that plots run without error, so I leave this 0
do_cluster = 0 #leave as 0 because cluster isn't working (except on colab), and typical recordings (size 128 x 256 x 20 x 3000) don't take that long
cluster_backend = 'ipyparallel' #irrelevant if do_cluster=0

env_path = sys.path

if (re.search("/Users/wienecke/", env_path[0])):
  pth_allrec = '/Users/wienecke/Documents/ambrose/stacks/'
  if do_denoise: #need gpu, don't have one locally
     raise Exception("no gpu, make do_denoise false")
elif (re.search("/home/caw846/", env_path[0])):
  pth_allrec = '/n/scratch3/users/c/caw846/stacks/'
  cluster_backend = 'SLURM' #this failed on O2, and so did cluster_backend = 'ipyparallel'
elif (re.search("/home/users/wienecke/", env_path[0])):
  pth_allrec = '/scratch/users/wienecke/stacks/'
elif (re.search('/content', env_path[0])):
  pth_allrec = '/content/drive/MyDrive/stacks/'
  do_cluster = 1 #cluster worked on colab
  index_extraction_param_set = None #not set up for arguments in colab

pth_super = '/'.join(pth_allrec.split('/')[:-2])
pth_denoising = os.path.join(pth_super, 'denoising')
if not os.path.exists(pth_denoising):
    os.mkdir(pth_denoising)
pth_denoised = os.path.join(pth_super, 'denoised')
if not os.path.exists(pth_denoised):
    os.mkdir(pth_denoised)


if len(sys.argv)>1:
  [index_extraction_param_set, region_extraction, do_motion_correction,
  do_denoise, use_denoised, do_extraction, do_planar_extraction,
  recdates,  fly, trial, do_cropping_session,
  recording_index] = parse_command_line(index_extraction_param_set = index_extraction_param_set,
                      region_extraction = region_extraction, do_motion_correction = do_motion_correction,
                      do_denoise = do_denoise, use_denoised = use_denoised, do_extraction = do_extraction, do_planar_extraction = do_planar_extraction,
                      recdates = recdates, fly = fly, trial = trial, do_cropping_session = do_cropping_session,
                      recording_index = recording_index)


print("STARTING EXTRACT.PY, STARTING EXTRACT.PY, STARTING EXTRACT.PY, STARTING EXTRACT.PY, STARTING EXTRACT.PY, STARTING EXTRACT.PY, STARTING EXTRACT.PY, STARTING EXTRACT.PY")
print(index_extraction_param_set)
print(region_extraction)
print(do_motion_correction)
print(do_denoise)
print(do_extraction)
print(do_planar_extraction)
print(recdates)
print(fly)
print(trial)
print(do_cropping_session)
print(recording_index)

countz = 0
for recording_date in recdates:

  tmpdate = datetime.datetime.now().strftime("%Y%m%dT%H%M%S")

  pth_fldrs_pattern = pth_allrec + recording_date + '-' + fly + '_*/'
  fn_pattern = recording_date + '-' + fly + '*_trial_00' + trial + '_*.tif'
  pth_fldrs = sorted(glob.glob(pth_fldrs_pattern))
  old_mat_files = 0
  if not pth_fldrs:  #if no matches try another filename pattern (files from previous project)
    old_mat_files = 1
    pth_fldrs_pattern = pth_allrec + recording_date + '_' + fly + '/'
    fn_pattern = recording_date + '_' + fly + '_' + trial + '_stackRaw_mc_.mat'
    pth_fldrs = sorted(glob.glob(pth_fldrs_pattern))

  for pth_fldr in pth_fldrs:

    print(pth_fldr)

    pth_allfiles = sorted(os.listdir(pth_fldr))

    for f in pth_allfiles:

      if fnmatch.fnmatch(f, fn_pattern):

        countz = countz + 1
        if recording_index==0 or (recording_index!=0 and countz==recording_index): #if 0, do all files, otherwise only file matching index

          pth_datafile = pth_fldr + f
          print(pth_datafile)

          if old_mat_files: #for my old project
            fn_prefix = f[:-5]
            pth_tif_reg_tmp = []
            pth_tif_reg = pth_datafile
            pth_tif_dn = pth_datafile[:-4] + 'dn_.tif'
          else:
            fn_prefix = f.split('_')[0].split('-')[0] + '_' + f.split('_')[0].split('-')[1]  + '_' + f.split('_')[-2][-1] #change hyphen to underscore
            pth_allrec_fnsave = pth_fldr + fn_prefix
            pth_tif_reg_tmp = [pth_allrec_fnsave + '_caimanregtmp_.tif']
            pth_tif_reg = [pth_allrec_fnsave + '_caimanreg_.tif']
            pth_tif_dn = [pth_allrec_fnsave + '_cmregcaddn_.tif']
            pth_tif_dn = [pth_allrec_fnsave + '_cmregcaddn_.tif']
            pth_md = [pth_allrec_fnsave + '_metadatanew_.mat']
            md = read_save_metadata(pth_datafile, pth_md)



In [ ]:
#@title load unregistered movie

n_processes = 1
dview = None

if do_motion_correction and not do_cropping_session:

    Y = imread(pth_datafile).astype('float32') ##having trouble on O2 with caiman function cm.load so just using imread from tifffile.tifffile

    Y = Y.reshape(md['dims'][0], md['dims'][1]+md['flyback'], md['dims'][2], md['dims'][3])

    Y = Y[:,:-md['flyback'],:,:] #crop md['flyback'] frames
    #Y[540:600,4,:,:].play(magnification=2) #play in order t z y x
    Y = np.transpose(Y, (0, 3, 2, 1)) #put in order t x y z

    min_mov = int(np.min(Y))
    Y = Y - min_mov #make movie nonnegative (not sure this is necessary)
    print("MIN BEFORE MOTION CORRECTION " + str(min_mov))

    if do_background_subtraction:
        
        for zind in np.arange(Y.shape[-1]): #for every z slice 

            dimorder = 'txy'
            pth_bgplot_save = pth_allrec_fnsave + '_' + str(zind)
            br = bgremover(Y[:,:,:,zind], pth_bgplot_save, half_wid=bg_patch_halfwidth, dimorder=dimorder)
            br.draw_patches()
            br.remove_bg()
            br.make_plots()
            Y[:,:,:,zind] = np.transpose(br.out, (0, 2, 1))
                
        min_mov = int(np.min(Y))
        Y = Y - min_mov #make movie nonnegative (not sure this is necessary)
        print("MIN BEFORE MOTION CORRECTION, AFTER BG SUB " + str(min_mov))
    
                
    imwrite(pth_tif_reg_tmp[0], Y) #write as t x y z


In [ ]:
#@title start cluster

if do_cluster:
  if 'dview' in locals(): cm.stop_server(dview=dview)
  cc, dview, n_processes = cm.cluster.setup_cluster(backend=cluster_backend, n_processes=None, single_thread=False)


In [ ]:
#@title register movie (motion correct)

if do_motion_correction and not do_cropping_session:

    opts_dict, indices_ex, fnadd = configs(index_extraction_param_set = None, fnames = pth_tif_reg_tmp, min_mov = min_mov, md = md) #configs for motion correction (will also define for extraction, but extraction params are in redefined later call to configs)
    opts = cnmf.params.CNMFParams(params_dict=opts_dict)

    #sys.setprofile(tracefunc)
    mc = cm.motion_correction.MotionCorrect(pth_tif_reg_tmp, dview=dview, **opts.get_group('motion'))
    mc.motion_correct(save_movie=True)
    os.remove(pth_tif_reg_tmp[0])

    border_to_0 = 0 if mc.border_nan == 'copy' else mc.border_to_0
    basename_memap = pth_tif_reg[0].split('/')[-1][:-4]
    pth_mmap_reg = cm.save_memmap(mc.mmap_file, base_name=basename_memap, order='C', border_to_0=border_to_0, dview=dview) # save in order C (motion_correct above has to save in order F)
    Y, dims_spatial, dim_time = cm.load_memmap(pth_mmap_reg)
    Y = np.reshape(Y.T, [dim_time] + list(dims_spatial), order='F')

    min_mov_after_reg = int(np.min(Y))
    Y = Y - min_mov_after_reg #make movie nonnegative for extraction (not sure this is necessary)
    print("MIN AFTER MOTION CORRECTION " + str(min_mov_after_reg))

    os.remove(mc.mmap_file[0]) #remove the mmap file in F order
    os.remove(pth_mmap_reg) #remove the mmap file in C order

    imwrite(pth_tif_reg[0], np.transpose(Y.astype('uint16'), (0, 3, 2, 1)).reshape(dim_time * dims_spatial[2], dims_spatial[1], dims_spatial[0])) #write the registered movie as tif (uint16) for use in matlab, and caiman extraction below




In [ ]:
#@title remove background and denoise (each z slice separately)

if do_denoise:

    %matplotlib inline
    
    # os.system("source /n/app/miniconda3/4.10.3/etc/profile.d/conda.sh; \
    # conda run -n deepcadrt ~/.conda/envs/deepcadrt/bin/python3 ~/scopa/denoise.py" \
    # + " --pth_in " + pth_tif_reg[0] \
    # + " --pth_out " + pth_tif_dn[0] \
    # + " --pth_denoising " + pth_denoising \
    # + " --pth_denoised " + pth_denoised \
    # + " --fn_prefix " + fn_prefix \
    # + " --dims " + ' '.join(map(str,  md['dims'])))

    pth_in = pth_tif_reg[0].copy()
    pth_out = pth_tif_dn.copy()
    pth_denoising = pth_denoising.copy()
    pth_denoised = pth_denoised.copy()
    fn_prefix = fn_prefix.copy()
    dims = md['dims'].copy()


    if pth_in.endswith( '.mat'):

        mat = mat73.loadmat(pth_in)
        Y = mat['stackRaw_mc']
        Y = np.moveaxis(Y, [0, 2], [2, 0]) #put in order t y x
        Y = Y[..., np.newaxis]

    else:

        Y = imread(pth_in).astype('float32')  
        Y = Y.reshape(dims)
        Y = np.transpose(Y, (0, 2, 3, 1)) #put in order t y x z (not t x y z)

        size_pre_denoise = Y.shape

        zind_all_dn = np.arange(Y.shape[-1])


    print(pth_in)
    for zii in zind_all_dn: #deepcad wants 3d data, so for each z slice (doing this rather than using all z slices in reshaped data because noise varies across z)

        Ynew = Y[:,:,:,zii]
        print("denoising slice " + str(zii))
        print(Y.shape)
        print(Ynew.shape)

        dnfolder = fn_prefix + '_' + str(zii)
        tifname = dnfolder + '_.tif'
        tiffolder_path = os.path.join(pth_denoising, dnfolder)
        testfolder_path = os.path.join(tiffolder_path, dnfolder + '_*')
        if os.path.exists(tiffolder_path): 
            shutil.rmtree(tiffolder_path) #just remove it because for some reason existing timestamped folders deepcad creates can cause error 
        os.mkdir(tiffolder_path)
        pth_tif_pdn = os.path.join(tiffolder_path, tifname)
        print(pth_tif_pdn)
        imwrite(pth_tif_pdn, Ynew.astype('float'), photometric='minisblack') #put the tif in the folder deepcad looks to for training data


        Lt, Ly, Lx = Ynew.shape
        print(Lt)
        print(Ly)
        print(Lx)

        n_epochs = 2                # number of training epochs
        GPU = '0'                   # the index of GPU you will use (e.g. '0', '0,1', '0,1,2')
        manual_max_dataset_size = 4000
        train_datasets_size = np.max([manual_max_dataset_size, int(np.ceil(Lt/4))])  # datasets size for training (how many 3D patches)
        patch_x = int(np.ceil(Lx/4)) # 
        patch_y = int(np.ceil(Ly/4)) #
        patch_t = 300                #
        overlap_factor = 0.4        # the overlap factor between two adjacent patches
        intensity_scale_factor = 1 # the factor for image intensity scaling
        num_workers = 0             # if you use Windows system, set this to 0.

        # Setup some parameters for result visualization during training period (optional)
        save_test_images_per_epoch = True  # whether to save result images after each epoch

        train_dict = {
            # dataset dependent parameters
            'patch_x': patch_x,                          # the width of 3D patches
            'patch_y': patch_y,                          # the height of 3D patches
            'patch_t': patch_t,                          # the time dimension (frames) of 3D patches
            'overlap_factor':overlap_factor,             # the factor for image intensity scaling
            'scale_factor': intensity_scale_factor,      # the factor for image intensity scaling
            'select_img_num': train_datasets_size,       # select the number of images used for training 
            'train_datasets_size': train_datasets_size,  # datasets size for training (how many 3D patches)
            'datasets_path': tiffolder_path,             # folder containing files for training
            'pth_dir': tiffolder_path,                   # the path for pth file and result images

            # network related parameters
            'n_epochs': n_epochs,                          # the number of training epochs
            'lr': 0.00005,                                 # learning rate
            'b1': 0.5,                                     # Adam: beta1
            'b2': 0.999,                                   # Adam: beta2
            'fmap': 8,  # was 16 by default                # model complexity
            'GPU': GPU,                                    # GPU index
            'num_workers': num_workers,                    # if you use Windows system, set this to 0.
            'visualize_images_per_epoch': False,                       # whether to show result images after each epoch
            'save_test_images_per_epoch': save_test_images_per_epoch,  # whether to save result images after each epoch
            'colab_display': False
        }

        tc = training_class(train_dict)
        tc.run()

        test_datasize = Lt
        para_path_tmp = glob.glob(os.path.join(testfolder_path, '*.yaml'))[0]
        denoise_model = para_path_tmp.split('/')[-2]
        para_path = glob.glob(os.path.join(tiffolder_path, denoise_model, '*.yaml'))[0]

        import yaml

        with open(para_path, "r") as stream:
            #para_dict = yaml.safe_load(stream)
            patch_t = train_dict['patch_t']
            patch_x = train_dict['patch_x']
            patch_y = train_dict['patch_y']
            fmap = train_dict['fmap']
            overlap_factor = train_dict['overlap_factor']

        test_dict = {
            # dataset dependent parameters
            'patch_x': patch_x,               # the width of 3D patches
            'patch_y': patch_y,               # the height of 3D patches
            'patch_t': patch_t,               # the time dimension (frames) of 3D patches
            'overlap_factor':overlap_factor,    # overlap factor
            'scale_factor': intensity_scale_factor,                  # the factor for image intensity scaling
            'test_datasize': test_datasize,     # the number of frames to be tested
            'datasets_path': tiffolder_path,     # folder containing all files to be tested
            'pth_dir': tiffolder_path,                 # pth file root path
            'denoise_model' : denoise_model,    # A folder containing all models to be tested
            'output_dir' : tiffolder_path,         # result file root path
            # network related parameters
            'fmap': fmap,                         # number of feature maps
            'GPU': GPU,                         # GPU index
            'num_workers': num_workers,         # if you use Windows system, set this to 0.
            'visualize_images_per_epoch': False,# whether to display inference performance after each epoch
            'save_test_images_per_epoch': True, # whether to save inference image after each epoch in pth path
            'colab_display': False
        }

        tc = testing_class(test_dict)
        tc.run()

        if tc.colab_display:
            display_filename = tc.result_display
            print('\033[1;31mDisplaying denoised file of the last epoch-----> \033[0m')
            print(display_filename)
            # normalize the image and display
            img = display_img(display_filename,norm_min_percent=1, norm_max_percent=99)
            plt.imshow(img,cmap=plt.cm.gray,vmin=0,vmax=255)
            plt.axis('off')
            plt.show()

        outtiff_path = glob.glob(os.path.join(tiffolder_path, 'DataFolderIs_*', 'E_0' + str(n_epochs) + '_*', '*output.tif'))[0]
        pth_destination = pth_denoised + '/' + outtiff_path.split('/')[-1]
        if os.path.isfile(pth_destination): #if completed file (for single z slice) exist from previous run, delete it (full denoised 4d recording is reassembled in pth_out)
            os.remove(pth_destination)
        shutil.move(outtiff_path, pth_denoised)
        shutil.rmtree(tiffolder_path)
        shutil.rmtree('/'.join(tiffolder_path.split('/')[:-1]) + '/bg_remove')



In [ ]:
#@title concatenate denoised movie in z and save

if do_denoise:

  Y = np.zeros(size_pre_denoise)
  pth_denoised_singles = glob.glob(pth_denoised + '/' + fn_prefix + '*')
  countz = 0
  for f in pth_denoised_singles: #loop over each denoised z slice and reassemble into array matching shape of original 4d volume  
      print(countz)
      Ynew = imread(f)
      print(Ynew.dtype)
      Y[:,:,:,countz] = Ynew
      countz+=1

  if countz!=size_pre_denoise[3]-1:
      raise Exception("not all slices present")

  min_mov = int(np.min(Y))
  Y = Y - min_mov #make nonnegative for extraction later (not sure this is necessary)
  print("MIN AFTER DENOISING " + str(min_mov))
  Y = Y.astype('uint16')
  Y = np.transpose(Y, (0, 3, 1, 2))
  print(Y.shape)
  Y = Y.reshape(size_pre_denoise[0] * size_pre_denoise[3], size_pre_denoise[1], size_pre_denoise[2])
  print(Y.shape)
  imwrite(pth_out, Y) #write the registered movie as tif for use in matlab, and caiman extraction below


In [ ]:
#@title load registered movie (if necessary) and create crop box for extraction

region_extraction = 'pb' #string not list of string (ie just do one region in ipynb bc loop is not really convenient in this notebook)
use_denoised = 1

if do_extraction:

    if use_denoised:
        pth_exin = pth_tif_dn
    else:
        pth_exin = pth_tif_reg

    Y = imread(pth_exin).astype('float32')
    Y = Y.reshape(md['dims'])
    Y = np.transpose(Y, (0, 3, 2, 1)) #put in order t x y z
    print(Y.shape)


    if region_extraction == '':

      do_cropping = 0
      limits_str = ''

    else:

      do_cropping = 1
      create_croplim = 0

      try:

          fn_croplim_pattern = pth_tif_reg[0][:-4] + region_extraction + '_croplim_.npy' #find file matching fov subregion with some crop lim
          fn_croplim = glob.glob(fn_croplim_pattern)
          if len(fn_croplim) > 1:
              raise Exception("too many crop files")
          with open(fn_croplim[0], 'rb') as fnc:
              croplim = np.load(fnc)
          limits_str = region_extraction + '_' + str(croplim[0]) + '_' + str(croplim[1]) + '_' + str(croplim[2]) + '_' + str(croplim[3]) + '_' + str(croplim[4]) + '_' + str(croplim[5]) + '_' + str(croplim[6]) + '_' + str(croplim[7])

      except:

          create_croplim = 1
          Ymt = np.mean(Y, axis = 0)
          zlimits = select_fov_z(Ymt)
          Ymtz = np.mean(Ymt[:,:,zlimits[0]:zlimits[1]], axis = 2)
          select_fov_xy(Ymtz)


In [ ]:
#@title crop fov for extraction

if do_extraction and do_cropping:  #had to put this here (in new cell) to get interactive plots to work in above cell

  if create_croplim:

    fn_crop_lim_xy = pth_tif_reg[0][:-4] + 'croplimxytmp_.npy'
    with open(fn_crop_lim_xy, 'rb') as fnc:
      croplimxy = np.load(fnc)
    ylimits = (croplimxy[0], croplimxy[1])
    xlimits = (croplimxy[2], croplimxy[3])
    tlimits = (1, dims_spacetime_original_noflyback[0])
    croplim = np.asarray((tlimits + xlimits + ylimits + zlimits)).astype(int) #make them 1-indexed for matlab later
    limits_str = region_extraction + '_' + str(croplim[0]) + '_' + str(croplim[1]) + '_' + str(croplim[2]) + '_' + str(croplim[3]) + '_' + str(croplim[4]) + '_' + str(croplim[5]) + '_' + str(croplim[6]) + '_' + str(croplim[7])
    fn_crop_lim = pth_tif_reg[0][:-4] + region_extraction + '_croplim_.npy'
    with open(fn_crop_lim, 'wb') as fncrop:
        np.save(fncrop, croplim)

  slt = slice(croplim[0]-1, croplim[1], 1) #croplim are 1-indexed, and the second/upper is not included
  slx = slice(croplim[2]-1, croplim[3], 1)
  sly = slice(croplim[4]-1, croplim[5], 1)
  slz = slice(croplim[6]-1, croplim[7], 1)
  indices_crop = [slt, slx, sly, slz]
  Ycrop = Y[tuple(indices_crop)]
  Y = None

else:

  raise Exception("no cropping requested, make sure Ycrop = Y.copy() works, or something similar")





In [ ]:
#@title start cluster

if 'dview' in locals(): cm.stop_server(dview=dview)
cc, dview, n_processes = cm.cluster.setup_cluster(backend='ipyparallel', n_processes=None, single_thread=False)

.............

In [ ]:
#@title memmap and load movie for extraction

if do_extraction:

    pth_tif_ex = [pth_tif_dn[0][:-4] + limits_str + '_caimanex_.tif']
    imwrite(pth_tif_ex[0], Ycrop) #must write it to memmap it, and must memmap it to use patches in extraction
    basename_memap = pth_tif_ex[0].split('/')[-1][:-4]
    border_to_0 = 0 #if mc.border_nan == 'copy' else mc.border_to_0
    fn_mmap_ex = cm.save_memmap(pth_tif_ex, base_name=basename_memap, order='C', border_to_0=border_to_0, dview=dview) # exclude borders
    os.remove(pth_tif_ex[0])
    Ycrop, dims_spatial, T = cm.load_memmap(fn_mmap_ex) #np.allclose(images, images2, rtol=1e-05, atol=1e-04, equal_nan=False)
    Ycrop = np.reshape(Ycrop.T, [T] + list(dims_spatial), order='F')


In [ ]:
#@title extract

if do_extraction:

  if index_extraction_param_set is None:
      index_extraction_param_set_new = [None] #make it iterable with brackets
  elif index_extraction_param_set<0: #if negative, initiate loop over extraction params here, range [0 - index_extraction_param_set]
      manual_start_ind = 0
      index_extraction_param_set_new = np.arange(manual_start_ind, -index_extraction_param_set)
  else: #if positive, just the one extraction param whose index matches index_extraction_param_set
      index_extraction_param_set_new = [index_extraction_param_set]

  print("index_extraction_param_set NEW " + str(index_extraction_param_set_new))

  for ii in index_extraction_param_set_new:

      try: #since some param sets will error

          opts_dict, indices_ex, fnadd = configs(index_extraction_param_set = ii, fnames = fn_mmap_ex, md = md, do_planar_extraction = do_planar_extraction, dims_spatial = dims_spatial) #param set for extraction
          opts = cnmf.params.CNMFParams(params_dict=opts_dict)

          if do_planar_extraction: #adjust images and some params for planar
              sliceindz = np.arange(Ycrop.shape[-1])
              dims_roimask_spatial = (dims_spatial[0], dims_spatial[1])
          else:
              sliceindz = [np.arange(Ycrop.shape[-1])] #all slices (not planar)
              dims_roimask_spatial = (dims_spatial[0], dims_spatial[1], dims_spatial[2])

          countz = 0
          for si in sliceindz: #for each slice (or all slices if do_planar_extraction = false)

              cnm = None
              cnm2 = None

              if do_planar_extraction:
                  print("PLANAR EXTRACTION FOR SLICE " + str(si))
                  images_sliced = Ycrop[:,:,:,si]
              else:
                  print("3D EXTRACTION FOR ALL SLICES")
                  images_sliced = Ycrop #can't .copy() for some reason, for 3d extraction keep all images for loop over ii

              if do_cluster:
                  if 'dview' in locals(): cm.stop_server(dview=dview)
                  cc, dview, n_processes = cm.cluster.setup_cluster(backend=cluster_backend, n_processes=None, single_thread=False)

              cnm = cnmf.CNMF(n_processes, params=opts, dview=dview)
              cnm = cnm.fit(images_sliced, indices = indices_ex)

              cnm.estimates.evaluate_components(images_sliced, cnm.params, dview=dview)
              print(('NUMGOOD ' + str(len(cnm.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm.estimates.idx_components_bad))))

              cnm.estimates.select_components(use_object=True, save_discarded_components=False)

      except: #since some param combos will error
          mdict = {}
          pth_mat_ex = [pth_tif_ex[0][:-5] + fnadd + 'rois_FAILURE_.mat']



In [ ]:
#@title start cluster

if 'dview' in locals(): cm.stop_server(dview=dview)
cc, dview, n_processes = cm.cluster.setup_cluster(backend='ipyparallel', n_processes=None, single_thread=False)


In [ ]:
#@title extract again

if do_extraction:

  try:

    cnm2 = cnm.refit(images_sliced)

    cnm2.estimates.evaluate_components(images_sliced, cnm2.params, dview=dview)
    print(('REFIT: NUMGOOD ' + str(len(cnm2.estimates.idx_components)) + ' NUMBAD ' + str(len(cnm2.estimates.idx_components_bad))))

    cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=False) #use_residuals=False to not include residuals in traces for dff computation (default)
    dff_residfalse = cnm2.estimates.F_dff
    cnm2.estimates.detrend_df_f(detrend_only=False, flag_auto=True, use_residuals=True) #use_residuals=True to include residuals in traces for dff computation
    dff_residtrue = cnm2.estimates.F_dff
    cnm2.estimates.select_components(use_object=True, save_discarded_components=False)

  except:

    mdict = {}
    pth_mat_ex = [pth_tif_ex[0][:-5] + fnadd + 'rois_FAILURE_.mat']


In [ ]:
#@title plots

if do_extraction:

  if do_planar_extraction:

      Cn = cm.local_correlations(images_sliced.transpose(1,2,0))
      Cn[np.isnan(Cn)] = 0
      print('you may need to change the data rate to generate nb_view_components: use jupyter notebook --NotebookApp.iopub_data_rate_limit=1.0e10 before opening jupyter notebook')

      cnm.estimates.plot_contours_nb(img=Cn) #img=None for mean projection

      cnm2.estimates.plot_contours_nb(img=Cn, idx=cnm2.estimates.idx_components)

      cnm2.estimates.nb_view_components(img=Cn, idx=cnm2.estimates.idx_components)

      if len(cnm2.estimates.idx_components_bad) > 0:
        cnm2.estimates.nb_view_components(img=Cn, idx=cnm2.estimates.idx_components_bad)
      else:
        print("No components were rejected.")

      cnm.estimates.play_movie(images_sliced, q_max=99.9, gain_res=2, magnification=2, bpx=True, include_bck=False, save_movie=True)

      A2 = cnm.estimates.A.toarray().reshape(opts.data['dims'] + (-1,), order='F').transpose([2, 0, 1])
      Nc = A2.shape[0]
      grid_shape = (np.ceil(np.sqrt(Nc/2)).astype(int), np.ceil(np.sqrt(Nc*2)).astype(int))
      plt.figure(figsize=np.array(grid_shape[::-1])*1.5)
      plt.imshow(montage(A2, rescale_intensity=True, grid_shape=grid_shape))
      plt.axis('off')

  else:

      cnm.estimates.nb_view_components_3d(image_type='mean', dims_spatial=dims_spatial, axis=2)



In [ ]:
#@title save

if do_extraction:

  if countz==0: #THIS IS POINTLESS WITHOUT THE LOOP, COUNTZ IS ALWAYS 0 (NO LOOP IN IPYNB CURRENTLY)

      padnum = 10
      padnum_b = 3
      numroi_stack_pad = cnm2.estimates.A.shape[-1] + padnum #extra since it can vary a little across fits (even above input k)
      dims_roimask_stack = ( dims_roimask_spatial + (numroi_stack_pad, ) )
      dims_roimask_b_stack = ( dims_roimask_spatial + (cnm2.estimates.b.shape[-1] + padnum_b, ) )
      dims_timeseries_stack = ( numroi_stack_pad, cnm2.estimates.C.shape[1] )

      stack_masks = np.zeros(dims_roimask_stack + (len(sliceindz), ) )
      stack_masks_b = np.zeros(dims_roimask_b_stack + (len(sliceindz), ) )
      stack_c = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
      stack_yra = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
      stack_s = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
      stack_df = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
      stack_dfr = np.zeros(dims_timeseries_stack + (len(sliceindz), ) )
      stack_snr = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
      stack_rval = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
      stack_idx = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )
      stack_idx_bad = np.zeros((dims_timeseries_stack[0], ) + (len(sliceindz), ) )

  numroi_slice = cnm2.estimates.A.shape[-1]
  numroi_b_slice = cnm2.estimates.b.shape[-1]
  dims_mask_slice = (dims_roimask_spatial + (numroi_slice, ) )
  dims_mask_b_slice = (dims_roimask_spatial + (numroi_b_slice, ) )

  stack_masks[..., :numroi_slice, countz] = np.reshape(cnm2.estimates.A.toarray(), dims_mask_slice, order='F')
  stack_masks_b[..., :numroi_b_slice, countz] = np.reshape(cnm2.estimates.b, dims_mask_b_slice, order='F')
  stack_c[:numroi_slice,:,countz] = cnm2.estimates.C
  stack_yra[:numroi_slice,:,countz] = cnm2.estimates.YrA
  stack_s[:numroi_slice,:,countz] = cnm2.estimates.S
  stack_df[:numroi_slice,:,countz] = dff_residfalse
  stack_dfr[:numroi_slice,:,countz] = dff_residtrue
  stack_snr[:numroi_slice,countz] = cnm2.estimates.SNR_comp
  stack_rval[:numroi_slice,countz] = cnm2.estimates.r_values
  #stack_idx[:cnm2.estimates.idx_components.shape[0],countz] = cnm2.estimates.idx_components
  #if cnm2.estimates.idx_components_bad.shape==(1,):
  #    stack_idx_bad[:cnm2.estimates.idx_components_bad.shape[0],countz] = cnm2.estimates.idx_components_bad

  countz = countz + 1

  log_files = glob.glob('*_LOG_*')
  for log_file in log_files:
      os.remove(log_file)

  mdict = {}
  mdict['roimasks'] = stack_masks.astype('float32')
  mdict['roimasks_b'] = stack_masks_b.astype('float32')
  mdict['C'] = stack_c.astype('float32')
  mdict['YrA'] = stack_yra.astype('float32')
  mdict['S'] = stack_s.astype('float32')
  mdict['dff'] = stack_df.astype('float32')
  mdict['dffr'] = stack_dfr.astype('float32')
  mdict['snr'] = stack_snr.astype('float32')
  mdict['rval'] = stack_rval.astype('float32')
  #mdict['idx'] = stack_idx
  #mdict['idxbad'] = stack_idx_bad

  if np.any(stack_masks):
      pth_mat_ex = [pth_tif_ex[0][:-5] + fnadd + 'rois_.mat']
  else:
      mdict = {}
      print("norois")
      pth_mat_ex = [pth_tif_ex[0][:-5] + fnadd + 'rois_NOROIS_.mat']

  sio.savemat(pth_mat_ex[0], mdict)

  if dview is not None: cm.stop_server(dview=dview)